# 11e The headroom rule on NOD, with measured noise ceilings

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 11a and 11b.

Development analysis on the nine donors only; the 21 targets are not read. The donors saw the same 120 COCO images in
10 runs, which gives every vertex a noise ceiling. Each donor in turn is the new person: borrowing from the other
eight, the low-rank map from k = 500 of its first ImageNet session, its own ridge at 500 and its own full model,
scored per vertex on the COCO images. The map's gain against ceiling and borrowing, and a split-half version. Each
run's COCO betas are fitted here from NOD's cleaned time series.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '11e-nodheadroom', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"donors": ["sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06", "sub-07", "sub-08", "sub-09"], "K": 200, "ks": [250, 500], "draws": [13, 14, 15], "folds": 5, "n_test": 200, "session_trials": 1000, "test_seed": 374, "cv_eval_share": 0.2, "cv_seed": 7, "thresholds": [0.05, 0.1], "primary_threshold": 0.05, "features": ["alexnet"], "wall_hours": 11.0, "encoder": "ridge", "min_vertices": 100, "k": 500, "half_seed": 384, "coco_runs": 10, "coco_images": 120, "bucket": "https://s3.amazonaws.com/openneuro.org", "dataset": "ds004496", "image_size": 256, "jpeg_quality": 95, "fit_images": 16540, "n_components": 1000, "batch": 64, "kpca_chunk": 2048, "min_align_r2": 0.99, "coco_tr": 241, "tr_s": 2.0, "lead_in_s": 16.0, "glm_alpha": 0.01, "max_ceiling_median": 0.99, "min_coco_runs": 8}')

import sys
import numpy as np
import pandas as pd
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C

IN = Path("/kaggle/input")
nod = next(p.parent for p in IN.rglob("sub-01_images.json"))
Y, IMG = {}, {}
for s in FIXED["donors"]:
    Y[s] = np.load(nod / f"{s}_betas.npy", mmap_mode="r")
    IMG[s] = np.array([n.replace(".JPEG", "") for n in json.loads((nod / f"{s}_images.json").read_text())])
V = Y[FIXED["donors"][0]].shape[1]
FEATS = {}
for name in FIXED["features"]:
    f = np.load(next(IN.rglob(f"11b_{name}_kpca1000.npz")))
    FEATS[name] = (f["z"].astype(np.float64), {str(n): j for j, n in enumerate(f["names"])})
feats = lambda name, names: FEATS[name][0][[FEATS[name][1][str(n)] for n in names]]
print({s: Y[s].shape for s in FIXED["donors"]}, "|", f"{(time.monotonic() - T0) / 60:.1f} min")

import re
NAME = re.compile(r"\d{12}\.jpg")
def coco_order(events, label_text):
    ev = events[events.trial_type.astype(float) < 1000]  # 2000 marks the catch trials
    pairs = {int(float(t)): str(c).split("/")[-1] for t, c in zip(ev.trial_type, ev.condition)}
    if sorted(pairs) != list(range(1, FIXED["coco_images"] + 1)):
        raise ValueError(f"image indices are not 1..{FIXED['coco_images']}")
    for t, c in zip(ev.trial_type, ev.condition):
        if pairs[int(float(t))] != str(c).split("/")[-1]:
            raise ValueError("one index, two images")
    names = [pairs[i] for i in range(1, FIXED["coco_images"] + 1)]
    if NAME.findall(label_text) != names:
        raise ValueError("label file order differs from the index order")
    return names

In [ ]:
import io
import subprocess
import urllib.error
import urllib.request
from PIL import Image
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel"], check=True)
import nibabel as nib
from cross_subject_encoding.features import streamed
from cross_subject_encoding.models import paper_linearizing as pl
B, D = FIXED["bucket"], FIXED["dataset"]
TMP = Path("/kaggle/tmp/coco")
TMP.mkdir(parents=True, exist_ok=True)

def get(url, tries=5):
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=300) as r:
                return r.read()
        except Exception as e:
            if i == tries - 1:
                raise
            print("  retry", url.split("/")[-1], type(e).__name__)
            time.sleep(15 * (i + 1))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nilearn"], check=True)
from nilearn.glm.first_level import make_first_level_design_matrix
from sklearn.linear_model import Ridge
MOTION = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]

def zrun_noise(b):
    # across-run variance of run-wise z-scored betas, mean over images (the ceiling's noise term)
    z = (b - b.mean(1, keepdims=True)) / (b.std(1, keepdims=True) + 1e-12)
    return z.var(0, ddof=1).mean(0)

def run_glm(s, r, events, base):
    # one run's betas as NOD's GLM.py 'separate' mode: cleaned surface series (hp128_s4), 0.5 s events after
    # the 16 s lead-in, SPM HRF, no drift terms, six motion regressors, ridge alpha 0.01 without intercept
    f = TMP / "ts.dtseries.nii"
    try:
        f.write_bytes(get(base + "_Atlas_hp128_s4.dtseries.nii", tries=2))
    except urllib.error.HTTPError as e:
        if e.code != 404:
            raise
        return None  # sub-01 run 2 has no cleaned series on OpenNeuro; that run is left out
    ts = np.asarray(nib.load(str(f)).get_fdata(dtype=np.float32))[:, :V]
    f.unlink()
    if ts.shape != (FIXED["coco_tr"], V) or not np.isfinite(ts).all():
        raise ValueError(f"{s} run {r}: time series {ts.shape}")
    ev = pd.DataFrame({"trial_type": [f"image{int(float(t)):03d}" for t in events.trial_type],
                       "onset": events.onset.to_numpy() + FIXED["lead_in_s"], "duration": events.duration.to_numpy()})
    dm = make_first_level_design_matrix(np.arange(FIXED["coco_tr"]) * FIXED["tr_s"], ev, drift_model=None, hrf_model="spm")
    conf = pd.read_csv(io.BytesIO(get(f"{B}/{D}/derivatives/fmriprep/{s}/ses-coco/func/{s}_ses-coco_task-coco_run-{r}_desc-confounds_timeseries.tsv")), sep="	")
    if len(conf) != FIXED["coco_tr"]:
        raise ValueError(f"{s} run {r}: {len(conf)} confound rows")
    x = np.column_stack([dm.to_numpy(), conf[MOTION].fillna(0).to_numpy()])
    coef = Ridge(alpha=FIXED["glm_alpha"], fit_intercept=False).fit(x, ts).coef_
    cols = [list(dm.columns).index(f"image{i:03d}") for i in range(1, FIXED["coco_images"] + 1)]  # row i = image index i + 1
    return coef[:, cols].T.astype(np.float32)

COCO_B, coco_names = {}, None
for s in FIXED["donors"]:
    runs, released = [], []
    for r in range(1, FIXED["coco_runs"] + 1):
        base = f"{B}/{D}/derivatives/ciftify/{s}/results/ses-coco_task-coco_run-{r}/ses-coco_task-coco_run-{r}"
        events = pd.read_csv(io.BytesIO(get(f"{B}/{D}/{s}/ses-coco/func/{s}_ses-coco_task-coco_run-{r:02d}_events.tsv")), sep="\t")
        names = coco_order(events, get(base + "_label.txt").decode())
        if coco_names is None:
            coco_names = names
        elif names != coco_names:
            raise ValueError(f"{s} run {r}: a different image set")
        f = TMP / "b.dscalar.nii"
        f.write_bytes(get(base + "_beta.dscalar.nii"))
        b = np.asarray(nib.load(str(f)).get_fdata(dtype=np.float32))
        if b.shape != (FIXED["coco_images"], V) or not np.isfinite(b).all():
            raise ValueError(f"{s} run {r}: beta {b.shape}")
        own = run_glm(s, r, events, base)
        if own is None:
            print(s, "run", r, "has no cleaned time series: left out", flush=True)
            continue
        released.append(b)
        runs.append(own)
    if len(runs) < FIXED["min_coco_runs"]:
        raise ValueError(f"{s}: only {len(runs)} COCO runs")
    COCO_B[s] = np.stack(runs)
    rel = np.stack(released)
    print(s, "COCO", COCO_B[s].shape, "| released betas: run 1 vs 2 r", round(float(np.corrcoef(rel[0].ravel(), rel[1].ravel())[0, 1]), 4),
          "| across-run noise after z-scoring, median: released", f"{np.median(zrun_noise(rel)):.2e}", "own GLM",
          f"{np.median(zrun_noise(COCO_B[s])):.3f}", "|", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)
    del rel, released
coco_dir = C.PROJECT_ROOT / "coco_images"
coco_dir.mkdir(exist_ok=True)
def coco_picture(n):
    # NOD lists the COCO file names but does not ship the pictures: fetched from COCO's own server
    for split in ("train2017", "val2017"):
        try:
            with urllib.request.urlopen(f"http://images.cocodataset.org/{split}/{n}", timeout=120) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code != 404:
                raise
    raise FileNotFoundError(n)
from PIL import ImageOps
for n in coco_names:
    img = ImageOps.fit(Image.open(io.BytesIO(coco_picture(n))).convert("RGB"), (FIXED["image_size"],) * 2, Image.BICUBIC)
    img.save(coco_dir / n, quality=FIXED["jpeg_quality"])  # centre square, 256 x 256, as the ImageNet stimuli in 11a

# features: 11b's fit images plus the 120 COCO images, kernel PCA refitted, then mapped onto 11b's components
f11b = np.load(next(IN.rglob("11b_alexnet_kpca1000.npz")))
fit_paths = [str(nod / "images" / f"{n}.jpg") for n in f11b["names"][f11b["fit_idx"]]]
paths = fit_paths + [str(coco_dir / n) for n in coco_names]
n_fit = len(fit_paths)
z_new, info = streamed.streamed_kpca(lambda rows: pl.extract([paths[r] for r in rows], "cpu", batch=FIXED["batch"], workers=2)[0],
                                     len(paths), np.arange(n_fit), "cpu", FIXED["n_components"], FIXED["kpca_chunk"])
z_old = f11b["z"][f11b["fit_idx"]].astype(np.float64)
W = np.linalg.lstsq(z_new[:n_fit].astype(np.float64), z_old, rcond=None)[0]
resid = z_old - z_new[:n_fit] @ W
align_r2 = 1 - (resid ** 2).sum(0) / ((z_old - z_old.mean(0)) ** 2).sum(0)
print("alignment R^2 per component: min", round(float(align_r2.min()), 5), "median", round(float(np.median(align_r2)), 5), "|",
      f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)
if align_r2.min() < FIXED["min_align_r2"]:
    raise ValueError("the refitted kernel PCA does not reproduce 11b's components")
Z_COCO = z_new[n_fit:].astype(np.float64) @ W
np.savez(C.PROJECT_ROOT / "11e_coco_alexnet_kpca1000.npz", names=np.array(coco_names), z=Z_COCO, align_r2=align_r2)

In [ ]:
def ols_coef(x, y):
    # ridge with a per-vertex penalty (5-fold CV on these images), returned as [intercept; coefficients] like OLS
    m = RidgePerOutput(lm.ALPHAS, FIXED["folds"], 13).fit(x, np.asarray(y, np.float32))
    print("   alpha median", float(np.median(m.best_alpha_)), "at grid edge", round(m.alpha_at_edge_, 3), flush=True)
    return np.vstack([(m.y_mean_ - m.x_mean_ @ m.coef_)[None], m.coef_]).astype(np.float32)
ols_pred = lambda beta, x: (np.concatenate([np.ones((len(x), 1)), x], 1) @ beta.astype(np.float64))
def r_cols(p, m):
    p, m = p - p.mean(0), m - m.mean(0)
    return (p * m).sum(0) / np.sqrt((p ** 2).sum(0) * (m ** 2).sum(0) + 1e-30)
BETA, RMAP = {}, {}
for s in FIXED["donors"]:
    y = np.asarray(Y[s], np.float32)
    for name in FIXED["features"]:
        BETA[(name, s)] = ols_coef(feats(name, IMG[s]), y)
    # vertex reliability for selection: the donor's own AlexNet model, fitted on 80% of its images, scored on 20%
    order = np.random.default_rng(FIXED["cv_seed"]).permutation(len(y))
    n_ev = int(FIXED["cv_eval_share"] * len(y))
    ev, fi = order[:n_ev], order[n_ev:]
    b = ols_coef(feats("alexnet", IMG[s][fi]), y[fi])
    RMAP[s] = r_cols(ols_pred(b, feats("alexnet", IMG[s][ev])), y[ev]).astype(np.float32)
    del y
    print(s, "encoders; cv r median / 99th pct / >0.05 / >0.1:", round(float(np.median(RMAP[s])), 4),
          round(float(np.percentile(RMAP[s], 99)), 4), int((RMAP[s] > 0.05).sum()), int((RMAP[s] > 0.1).sum()), "|", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)

In [ ]:
def r_vert(p, m):
    p, m = p - p.mean(0), m - m.mean(0)
    return (p * m).sum(0) / np.sqrt((p ** 2).sum(0) * (m ** 2).sum(0) + 1e-30)

def ceiling_r(b):
    # NSD's noise ceiling for the mean of n runs, from run-wise z-scored betas (Allen et al. 2022), as a correlation
    n = len(b)
    noise = b.var(0, ddof=1).mean(0)
    signal = np.clip(1 - noise, 0, None)
    snr2 = signal / np.maximum(noise, 1e-12)
    return np.sqrt(snr2 / (snr2 + 1 / n))

vert_rows, per_vertex = [], {}
for p, s in enumerate(FIXED["donors"], start=1):
    t = time.monotonic()
    others = [d for d in FIXED["donors"] if d != s]
    cols = np.where(np.mean([RMAP[d] for d in others], 0) > FIXED["primary_threshold"])[0]
    b = COCO_B[s][:, :, cols].astype(np.float64)
    b = (b - b.mean(1, keepdims=True)) / (b.std(1, keepdims=True) + 1e-12)  # z-score each run over the 120 images
    meas = b.mean(0)
    half = np.random.default_rng([FIXED["half_seed"], p]).permutation(FIXED["coco_images"])
    ha, hb = half[:len(half) // 2], half[len(half) // 2:]
    pred_avg = np.mean([ols_pred(BETA[("alexnet", d)][:, cols], Z_COCO) for d in others], 0)
    pred_own_full = ols_pred(BETA[("alexnet", s)][:, cols], Z_COCO)
    session1 = np.arange(FIXED["session_trials"])
    x_pool = feats("alexnet", IMG[s][session1])
    y_pool = np.asarray(Y[s][session1][:, cols], np.float64)
    avg_pool = np.mean([ols_pred(BETA[("alexnet", d)][:, cols], x_pool) for d in others], 0)
    mean, basis = lm.prediction_basis(avg_pool, FIXED["K"])
    maps, owns = [], []
    for d in FIXED["draws"]:
        idx = np.random.default_rng([d, p]).permutation(len(session1))[:FIXED["k"]]
        maps.append(lm.apply_lowrank_map(lm.fit_lowrank_map(avg_pool[idx], y_pool[idx], mean, basis, seed=d), pred_avg, mean, basis))
        owns.append(RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(x_pool[idx], y_pool[idx]).predict(Z_COCO))
    preds = {"avg": pred_avg, "map": np.mean(maps, 0), "own_k": np.mean(owns, 0), "own_full": pred_own_full}
    out = {"vertex": cols, "ceiling": ceiling_r(b), "ceiling_a": ceiling_r(b[:, ha]), "ceiling_b": ceiling_r(b[:, hb])}
    if np.median(out["ceiling"]) > FIXED["max_ceiling_median"]:
        raise ValueError(f"{s}: median ceiling {np.median(out['ceiling']):.4f}; the runs are not independent repeats")
    for name, pr in preds.items():
        out[name] = r_vert(pr, meas)
        out[name + "_a"], out[name + "_b"] = r_vert(pr[ha], meas[ha]), r_vert(pr[hb], meas[hb])
    per_vertex[s] = out
    np.savez(C.METRICS_DIR / f"11e_vertices_{s}.npz", **out)
    vert_rows.append({"subject": s, "vertices": int(len(cols)), "ceiling_median": float(np.median(out["ceiling"])),
                      **{f"{m}_r": float(np.mean(out[m])) for m in preds}})
    print(s, vert_rows[-1], f"{time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)
pd.DataFrame(vert_rows).to_csv(C.METRICS_DIR / "11e_people.csv", index=False)

In [ ]:
from scipy import stats
def fit(gain, ceil, borrow):
    x = np.column_stack([np.ones_like(ceil), ceil, borrow])
    coef, *_ = np.linalg.lstsq(x, gain, rcond=None)
    r2 = 1 - ((gain - x @ coef) ** 2).sum() / ((gain - gain.mean()) ** 2).sum()
    return {"intercept": float(coef[0]), "ceiling": float(coef[1]), "borrowing": float(coef[2]), "r2": float(r2)}
res = {"post_hoc": "development on NOD donors", "per_person": {}, "pooled": {}}
pool = {k: [] for k in ("gain", "own_gain", "ceil", "borrow", "gain_b", "head_a", "own_gain_b")}
for s, o in per_vertex.items():
    gain, own_gain = o["map"] - o["avg"], o["own_full"] - o["avg"]
    gain_b, head_a, own_gain_b = o["map_b"] - o["avg_b"], o["ceiling_a"] - o["avg_a"], o["own_full_b"] - o["avg_b"]
    res["per_person"][s] = {"map_fit": fit(gain, o["ceiling"], o["avg"]), "own_fit": fit(own_gain, o["ceiling"], o["avg"]),
                            "corr_gain_headroom": float(np.corrcoef(gain, o["ceiling"] - o["avg"])[0, 1]),
                            "split_half_corr": float(np.corrcoef(gain_b, head_a)[0, 1]),
                            "split_half_spearman": float(stats.spearmanr(gain_b, head_a).statistic),
                            "mean_gain": float(gain.mean()), "mean_headroom": float((o["ceiling"] - o["avg"]).mean())}
    for k, v in (("gain", gain), ("own_gain", own_gain), ("ceil", o["ceiling"]), ("borrow", o["avg"]),
                 ("gain_b", gain_b), ("head_a", head_a), ("own_gain_b", own_gain_b)):
        pool[k].append(v)
P = {k: np.concatenate(v) for k, v in pool.items()}
res["pooled"] = {"map_fit": fit(P["gain"], P["ceil"], P["borrow"]), "own_fit": fit(P["own_gain"], P["ceil"], P["borrow"]),
                 "corr_gain_headroom": float(np.corrcoef(P["gain"], P["ceil"] - P["borrow"])[0, 1]),
                 "split_half_corr": float(np.corrcoef(P["gain_b"], P["head_a"])[0, 1]),
                 "split_half_corr_own": float(np.corrcoef(P["own_gain_b"], P["head_a"])[0, 1]),
                 "vertices": int(len(P["gain"]))}
sh = np.array([v["split_half_corr"] for v in res["per_person"].values()])
res["split_half_positive"] = int((sh > 0).sum())
res["people"] = vert_rows
# headroom bins: mean map gain by decile of split-half headroom (pooled)
edges = np.quantile(P["head_a"], np.linspace(0, 1, 11))
bins = np.clip(np.digitize(P["head_a"], edges[1:-1]), 0, 9)
res["gain_by_headroom_decile"] = [{"headroom_a": float(P["head_a"][bins == i].mean()), "gain_b": float(P["gain_b"][bins == i].mean()),
                                   "own_gain_b": float(P["own_gain_b"][bins == i].mean())} for i in range(10)]
(C.METRICS_DIR / "11e_summary.json").write_text(json.dumps(res, indent=2))
print("11E_SUMMARY_JSON " + json.dumps(res))